# 02. 서울 전월세 거래량 머신러닝

**목표:** 현재 월의 시장·인구·금리 정보를 이용하여 **자치구별 다음 달 전월세 거래건수(`nextContractCount`)**를 예측합니다.

모델은 단순 기준선(Baseline), Linear Regression, Random Forest를 동일한 시간순 테스트 구간에서 비교합니다.

> 선행 조건: `01_data_pipeline_clean.ipynb`를 먼저 실행하여 MySQL의 `ml_rent_market` 테이블을 생성해야 합니다.

## 1. 데이터 로드
MySQL의 최종 통합 테이블을 읽고 데이터 구조와 결측치를 확인합니다.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

load_dotenv("../.env", override=True)
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM ml_rent_market", engine)
print(df.shape)
df.head()

In [ ]:
print(df.isnull().sum())

## 2. 시간 변수와 예측 Target 생성
각 자치구에서 현재 월의 다음 행을 `nextContractCount`로 이동(`shift(-1)`)하여 다음 달 거래건수를 target으로 만듭니다. 12월은 다음 달 데이터가 없으므로 학습 대상에서 제외됩니다.

In [ ]:
df["yearMonth"] = pd.to_datetime(df["yearMonth"])
df["year"] = df["yearMonth"].dt.year
df["month"] = df["yearMonth"].dt.month

df = df.sort_values(["guName", "yearMonth"]).copy()
df["nextContractCount"] = df.groupby("guName")["contractCount"].shift(-1)
model_df = df.dropna(subset=["nextContractCount"]).copy()

print("모델링 데이터:", model_df.shape)
model_df[["guName", "yearMonth", "contractCount", "nextContractCount"]].head(12)

## 3. Feature 및 시간순 Train/Test 분리

랜덤 분할은 미래 데이터가 과거 학습에 섞이는 시계열 누수(leakage)를 만들 수 있으므로 사용하지 않습니다.

- Train: 1~9월 행 → 다음 달(2~10월) 예측
- Test: 10~11월 행 → 다음 달(11~12월) 예측

현재 데이터는 **2024년 한 해뿐인 275개 모델링 행**이므로 성능을 장기간 일반화된 결과로 해석하지 않습니다.

In [ ]:
features = [
    "contractCount", "moveIn", "moveOut", "netMove",
    "baseRate", "avgLivingPop", "month"
]

train = model_df[model_df["month"] <= 9].copy()
test = model_df[model_df["month"] >= 10].copy()

X_train, y_train = train[features], train["nextContractCount"]
X_test, y_test = test[features], test["nextContractCount"]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 4. Baseline
가장 단순한 기준은 **“다음 달 거래량 = 현재 달 거래량”**입니다. ML 모델은 최소한 이 기준과 비교해야 합니다.

In [ ]:
def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

baseline_pred = X_test["contractCount"]
baseline_metrics = metrics(y_test, baseline_pred)
baseline_metrics

## 5. Linear Regression
`netMove = moveIn - moveOut`이므로 세 변수를 동시에 넣으면 완전한 선형 종속성이 생깁니다. 따라서 선형회귀에서는 `netMove`를 제외합니다.

In [ ]:
lr_features = ["contractCount", "moveIn", "moveOut", "baseRate", "avgLivingPop", "month"]
X_train_lr = train[lr_features]
X_test_lr = test[lr_features]

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train)
lr_pred = lr_model.predict(X_test_lr)
lr_metrics = metrics(y_test, lr_pred)
lr_metrics

In [ ]:
# 계수는 변수 단위가 서로 다르므로 절대값만으로 중요도를 비교하지 않습니다.
coef_df = pd.DataFrame({
    "feature": lr_features,
    "coefficient": lr_model.coef_
}).sort_values("coefficient", key=abs, ascending=False)
coef_df

## 6. Random Forest
비선형 관계를 학습할 수 있는 앙상블 회귀 모델입니다. `random_state=42`로 재현성을 고정합니다.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_metrics = metrics(y_test, rf_pred)
rf_metrics

In [ ]:
rf_importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)
rf_importance

## 7. 모델 성능 비교
MAE/RMSE는 낮을수록, R²는 높을수록 테스트 데이터에 대한 예측 오차가 작습니다. 단, 2024년 한 해만 사용한 결과라는 한계가 있습니다.

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Baseline", **baseline_metrics},
    {"Model": "Linear Regression", **lr_metrics},
    {"Model": "Random Forest", **rf_metrics}
])
comparison

In [ ]:
comparison.set_index("Model")[["MAE", "RMSE"]].plot(kind="bar", figsize=(8, 5))
plt.title("Model Performance Comparison")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. 실제값과 예측값 확인
Random Forest 예측을 자치구·월별 실제값과 함께 확인합니다.

In [ ]:
result = test[["guName", "yearMonth"]].copy()
result["actual"] = y_test.values
result["predicted"] = rf_pred
result["error"] = result["actual"] - result["predicted"]
result.head(10)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(len(result)), result["actual"], label="Actual")
plt.plot(range(len(result)), result["predicted"], label="Predicted")
plt.xlabel("Test Sample")
plt.ylabel("Contract Count")
plt.title("Actual vs Predicted Rental Contract Count")
plt.legend()
plt.tight_layout()
plt.show()

## 9. 해석 및 한계

이 프로젝트의 현재 실행 결과에서는 Baseline이 Linear Regression과 Random Forest보다 높은 테스트 성능을 보였습니다. 이는 2024년 자료에서 **직전 월 거래량 자체가 다음 달 거래량의 강한 기준**이었음을 보여줍니다. ML을 사용했다는 사실보다 Baseline과 비교하여 추가 변수가 실제로 예측력을 개선했는지 검증하는 것이 중요합니다.

또한 다음 사항을 고려해야 합니다.
- 데이터 기간이 2024년 1년뿐이어서 계절성과 금리 변화의 장기 효과를 충분히 학습하기 어렵습니다.
- 기준금리는 학습 구간에서 변동이 제한적이어서 Random Forest 중요도가 0에 가깝게 나타날 수 있습니다.
- `moveIn`, `moveOut`, `netMove`는 서로 종속적이므로 선형 모델 해석 시 중복 사용을 피했습니다.
- 현재 월 인구이동·생활인구를 이용하므로 이 모델은 **현재 월 데이터가 확정된 뒤 다음 달을 예측하는 시점**을 가정합니다.

향후 2020~2025년처럼 기간을 확장하면 여러 금리 국면과 계절 패턴을 포함해 더 신뢰도 높은 시계열 검증이 가능합니다.